<a href="https://colab.research.google.com/github/mohanasudhashanmugam/DeepLearning/blob/main/Modified_train_theft_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!kaggle datasets download -d kipshidze/shoplifting-video-dataset
!unzip -q shoplifting-video-dataset.zip -d ./local_colab_storage

Dataset URL: https://www.kaggle.com/datasets/kipshidze/shoplifting-video-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
shoplifting-video-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
replace ./local_colab_storage/normal/normal-1.mp4? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [9]:
!ls /content/local_colab_storage/normal | wc -l

90


In [10]:
!ls /content/local_colab_storage/shoplifting | wc -l

92


In [11]:
import cv2
import os
import numpy as np
import argparse
from imutils import paths
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [12]:
# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()

ap.add_argument("-d", "--dataset",
	default="/content/local_colab_storage/",
	help="path to input dataset")
args_namespace, unknown = ap.parse_known_args()


# Convert to a dictionary
args = vars(args_namespace)

In [19]:
video_extensions = (
    ".mp4",
    ".avi",
    ".mkv",
    ".mov",
    ".wmv",
    ".flv",
    ".webm"
)

video_paths = list(paths.list_files(
    args["dataset"],
    validExts=video_extensions
))

video_labels = []

for video_path in video_paths:
    label = video_path.split(os.path.sep)[-2]
    video_labels.append(label)

print("Total videos:", len(video_paths))
print("Labels:", set(video_labels))

Total videos: 182
Labels: {'shoplifting', 'normal'}


In [20]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(video_labels)

print("Classes:", label_encoder.classes_)
print("Encoded labels:", y)

Classes: ['normal' 'shoplifting']
Encoded labels: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [21]:
from sklearn.model_selection import train_test_split

train_paths, test_paths, train_labels, test_labels = train_test_split(
    video_paths,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training videos:", len(train_paths))
print("Testing videos:", len(test_paths))

Training videos: 145
Testing videos: 37


In [22]:
import tensorflow as tf

class VideoDataGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        video_paths,
        labels,
        batch_size=4,
        max_frames=32,
        resize_dim=(224, 224),
        shuffle=True
    ):
        self.video_paths = video_paths
        self.labels = labels
        self.batch_size = batch_size
        self.max_frames = max_frames
        self.resize_dim = resize_dim
        self.shuffle = shuffle

        self.indices = np.arange(len(self.video_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.video_paths) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def process_video(self, video_path):

        cap = cv2.VideoCapture(video_path)

        total_frames = int(
            cap.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        if total_frames <= 0:
            cap.release()
            return np.zeros(
                (self.max_frames, *self.resize_dim, 3),
                dtype=np.float32
            )

        frame_indices = np.linspace(
            0,
            total_frames - 1,
            self.max_frames,
            dtype=int
        )

        video_frames = []

        for frame_idx in frame_indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                frame_idx
            )

            success, frame = cap.read()

            if not success:
                break

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            frame = cv2.resize(
                frame,
                self.resize_dim
            )

            frame = frame.astype(
                np.float32
            ) / 255.0

            video_frames.append(frame)

        cap.release()

        # Make sure every video has exactly 32 frames
        while len(video_frames) < self.max_frames:

            video_frames.append(
                video_frames[-1].copy()
                if video_frames
                else np.zeros(
                    (*self.resize_dim, 3),
                    dtype=np.float32
                )
            )

        return np.array(
            video_frames,
            dtype=np.float32
        )

    def __getitem__(self, index):

        start = index * self.batch_size
        end = min(
            start + self.batch_size,
            len(self.video_paths)
        )

        batch_indices = self.indices[start:end]

        X_batch = []
        y_batch = []

        for i in batch_indices:

            video = self.process_video(
                self.video_paths[i]
            )

            X_batch.append(video)
            y_batch.append(self.labels[i])

        X_batch = np.array(
            X_batch,
            dtype=np.float32
        )

        y_batch = tf.keras.utils.to_categorical(
            y_batch,
            num_classes=2
        )

        return X_batch, y_batch

In [23]:
train_generator = VideoDataGenerator(
    train_paths,
    train_labels,
    batch_size=2,
    max_frames=32,
    resize_dim=(224, 224),
    shuffle=True
)

test_generator = VideoDataGenerator(
    test_paths,
    test_labels,
    batch_size=2,
    max_frames=32,
    resize_dim=(224, 224),
    shuffle=False
)

In [24]:
X_batch, y_batch = train_generator[0]

print("Batch X shape:", X_batch.shape)
print("Batch y shape:", y_batch.shape)
print("Batch X dtype:", X_batch.dtype)
print("Batch X memory:",
      X_batch.nbytes / (1024**2),
      "MB")

Batch X shape: (2, 32, 224, 224, 3)
Batch y shape: (2, 2)
Batch X dtype: float32
Batch X memory: 36.75 MB


In [27]:

from tensorflow.keras import layers, models

aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomContrast(0.1)
])

# 1. Base CNN to extract features from a single frame
# We use MobileNetV2 because it is lightweight and fast

base_cnn = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_cnn.trainable = False  # Freeze weights as we don't destroy pre-trained weights by new training data during backpropagation

# Flatten the CNN output to a vector
pooling_layer = layers.GlobalAveragePooling2D()(base_cnn.output)
feature_extractor = models.Model(
    inputs=base_cnn.input,
    outputs=pooling_layer
)
# 2. Complete Video Model
video_input = layers.Input(
    shape=(32, 224, 224, 3)
)
# TimeDistributed applies the CNN to all frames individually
augmented_input = layers.TimeDistributed(aug)(video_input)
encoded_frames = layers.TimeDistributed(
    feature_extractor
)(augmented_input)

# LSTM tracks the movement across the timeline
x = layers.LSTM(
    64,
    dropout=0.5
)(encoded_frames)

x = layers.Dense(
    32,
    activation='relu'
)(x)

# # Output layer: Sigmoid activation for binary classification (0 or 1)
output = layers.Dense(
    2,
    activation='sigmoid'
)(x)

model = models.Model(
    inputs=video_input,
    outputs=output
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 32, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 32, 224, 224,   │             0 │
│ (TimeDistributed)               │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 32, 1280)       │     2,257,984 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │       344,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,604,450 (9.94 MB)

 Trainable params: 346,466 (1.32 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [28]:
## Run training for 30 to 50 epochs, but use an EarlyStopping callback.
#This tells Colab to keep training as long as the validation loss is improving, and automatically stop if it plateaus, saves time.

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 532s 5s/step - accuracy: 0.5724 - loss: 0.7018 - val_accuracy: 0.4865 - val_loss: 0.7161
Epoch 2/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 323s 4s/step - accuracy: 0.4966 - loss: 0.7135 - val_accuracy: 0.4865 - val_loss: 0.7092
Epoch 3/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 313s 4s/step - accuracy: 0.4759 - loss: 0.7086 - val_accuracy: 0.5405 - val_loss: 0.7055
Epoch 4/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 334s 5s/step - accuracy: 0.5517 - loss: 0.7000 - val_accuracy: 0.5405 - val_loss: 0.7034
Epoch 5/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 309s 4s/step - accuracy: 0.4759 - loss: 0.7060 - val_accuracy: 0.5676 - val_loss: 0.7029
Epoch 6/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 329s 4s/step - accuracy: 0.5655 - loss: 0.6990 - val_accuracy: 0.5405 - val_loss: 0.7020
Epoch 7/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 312s 4s/step - accuracy: 0.5931 - loss: 0.6773 - val_accuracy: 0.5135 - val_loss: 0.7006
Epoch 8/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 320s 4s/step - accuracy: 0.6000 - loss: 0.6867 - val_accuracy: 0.5405 - v

In [30]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Get raw probability predictions (numbers between 0.0 and 1.0)
predictions = model.predict(test_generator)
print(predictions)

19/19 ━━━━━━━━━━━━━━━━━━━━ 161s 6s/step
[[0.54963344 0.6037677 ]
 [0.48225722 0.50523436]
 [0.4993777  0.52102983]
 [0.36191326 0.48536333]
 [0.48785412 0.49098718]
 [0.48412967 0.4791139 ]
 [0.4556948  0.45533463]
 [0.47806802 0.5181623 ]
 [0.540833   0.5500725 ]
 [0.5275382  0.4973968 ]
 [0.504232   0.52445066]
 [0.49014053 0.49468398]
 [0.54739994 0.56327623]
 [0.4679567  0.58604527]
 [0.38336203 0.50994074]
 [0.41075546 0.5026858 ]
 [0.49778357 0.46246767]
 [0.51043266 0.5547594 ]
 [0.5537902  0.55059284]
 [0.4389993  0.46724334]
 [0.46222535 0.44930795]
 [0.3604119  0.4524356 ]
 [0.49537542 0.54690367]
 [0.51026064 0.5258819 ]
 [0.4236521  0.48526695]
 [0.54256195 0.5329972 ]
 [0.4223151  0.4681161 ]
 [0.4578029  0.49773568]
 [0.50777024 0.5751678 ]
 [0.43342382 0.46514618]
 [0.5431767  0.556928  ]
 [0.5463144  0.57539815]
 [0.48480764 0.4682254 ]
 [0.48042175 0.5554776 ]
 [0.49927437 0.52243835]
 [0.36364305 0.42745736]
 [0.45123428 0.54715085]]


In [36]:
# 2. Convert probabilities to binary choices: if > 0.5, it's Shoplifting (1), else Normal (0)
##binary_predictions = (predictions > 0.5).astype(int)

binary_predictions = np.argmax(predictions, axis=1)

# ============================================================
# TO GET TRUE TEST LABELS
#
# test_generator labels correspond to the video order.
# ============================================================

true_labels = []

for i in range(len(test_generator)):

    _, batch_labels = test_generator[i]

    true_labels.extend(
        batch_labels.astype(int)
    )


true_labels = np.array(
    true_labels
)

# Convert y_test from 2 columns to 1 column
true_labels = np.argmax(true_labels, axis=1)

print(binary_predictions)

print("true_labels", true_labels)

[1 1 1 1 1 0 0 1 1 0 1 1 1 1 1 1 0 1 0 1 0 1 1 1 1 0 1 1 1 1 1 1 0 1 1 1 1]
true_labels [0 0 0 0 1 0 0 1 1 0 1 1 0 0 0 1 0 1 0 0 1 1 0 0 1 1 1 1 1 0 1 0 1 1 1 1 0]


In [37]:
print("--- Confusion Matrix ---")
print(confusion_matrix(true_labels, binary_predictions))

--- Confusion Matrix ---
[[ 5 13]
 [ 3 16]]


In [38]:
print("\n--- Classification Report ---")
print(classification_report(true_labels, binary_predictions, target_names=['Normal', 'Shoplifting']))


--- Classification Report ---
              precision    recall  f1-score   support

      Normal       0.62      0.28      0.38        18
 Shoplifting       0.55      0.84      0.67        19

    accuracy                           0.57        37
   macro avg       0.59      0.56      0.53        37
weighted avg       0.59      0.57      0.53        37



In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

##model.save("shoplifting_detection_final.keras")
#print("\nFinal model saved successfully.")